In [ ]:
import pandas as pd
import csv
from Bio import SeqIO
import os

# ==============================================================================
# 1. 配置与准备
# ==============================================================================
# 定义氨基酸序列的字符集
AA = 'ACDEFGHIKLMNPQRSTVWY'
GAP_MAX = 5
THRESHOLDS = ['0.4', '0.5', '0.6', '0.7']

# 预测结果文件路径 (包含 Label)
PREDICTION_CSV = "predictions_5551_ensemble.csv"

# 输出目录
OUTPUT_DIR = "./"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ==============================================================================
# 2. 加载标签映射字典
# ==============================================================================
print(f"正在加载标签文件: {PREDICTION_CSV} ...")
if not os.path.exists(PREDICTION_CSV):
    raise FileNotFoundError(f"未找到标签文件: {PREDICTION_CSV}")

df_labels = pd.read_csv(PREDICTION_CSV)
df_labels['ID'] = df_labels['ID'].astype(str).str.strip()
id_to_label = dict(zip(df_labels['ID'], df_labels['Prediction']))

print(f"标签加载完成，共 {len(id_to_label)} 个 ID。")

# ==============================================================================
# 3. 循环处理不同阈值的 FASTA 文件
# ==============================================================================

# --- 【修改1】表头 Label 放第一位 ---
aaPairs = [aa1 + aa2 for aa1 in AA for aa2 in AA]
feature_names = []
for g in range(GAP_MAX + 1):
    for pair in aaPairs:
        feature_names.append(f'{pair}.gap{g}')

# 最终表头：Label 在最前
header = ['Label'] + feature_names 

for name in THRESHOLDS:
    fasta_file = f"./cat_{name}.fasta"
    output_csv = os.path.join(OUTPUT_DIR, f'encoded_cksaap_{name}_with_label.csv')
    
    if not os.path.exists(fasta_file):
        print(f"跳过: 文件未找到 {fasta_file}")
        continue
        
    print(f"\n正在处理: {fasta_file} ...")
    
    sequences = SeqIO.parse(fasta_file, "fasta")
    processed_count = 0
    skipped_count = 0
    
    with open(output_csv, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(header) # 写入表头
        
        for record in sequences:
            seq_id = str(record.id).strip()
            sequence = str(record.seq).upper()
            
            # 查找 Label
            if seq_id in id_to_label:
                label = id_to_label[seq_id]
            else:
                parts = seq_id.split('|')
                found = False
                for part in parts:
                    if part in id_to_label:
                        label = id_to_label[part]
                        found = True
                        break
                if not found:
                    skipped_count += 1
                    continue

            # 检查非法字符
            if any(c not in AA for c in sequence):
                skipped_count += 1
                continue

            # CKSAAP 计算
            features = []
            for g in range(GAP_MAX + 1):
                myDict = {pair: 0 for pair in aaPairs}
                total_pairs = 0
                
                # 统计频次
                for index1 in range(len(sequence) - g - 1):
                    index2 = index1 + g + 1
                    aa1 = sequence[index1]
                    aa2 = sequence[index2]
                    myDict[aa1 + aa2] += 1
                    total_pairs += 1
                
                # 计算频率
                for pair in aaPairs:
                    if total_pairs > 0:
                        features.append(myDict[pair] / total_pairs)
                    else:
                        features.append(0)
            
            # --- 【修改2】构建写入行：Label + 特征 ---
            # 将 label 放在列表的第一个位置
            row_to_write = [label] + features
            
            writer.writerow(row_to_write)
            processed_count += 1

    print(f"  -> 已保存至: {output_csv}")
    print(f"  -> 成功编码: {processed_count} 条")
    if skipped_count > 0:
        print(f"  -> 跳过: {skipped_count} 条")

print("\n所有任务完成。")